# Q&A across documents with LangChain and LangSmith

## What this notebook does

This notebook builds a Retrieval-Augmented Generation (RAG) chatbot that can answer questions across multiple documents. It starts by loading local source material from DOCX, PDF, and TXT files, with no internet access needed,, splitting that material into chunks, embedding the chunks, and storing them in a Chroma vector database.

After ingestion, the notebook shows how to query the vector store directly, then how to connect retrieval to a LangChain RAG chain so an LLM can synthesize answers from the retrieved context. The final section adds chat message history, allowing the chatbot to handle follow-up questions with conversational memory.

The notebook can run with OpenAI, Ollama, Gemini, or DeepSeek. The provider and model settings are loaded from the project-root `.env` file when available. DeepSeek is used for chat while embeddings default to Gemini because DeepSeek does not expose an embeddings API.


In [ ]:
# Select the LLM provider: "openai", "ollama", "gemini", or "deepseek".
# Values are loaded from the project-root .env file when present.
import os
import getpass
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


class ChatDeepSeek(ChatOpenAI):
    """Use function calling for structured output; DeepSeek rejects json_schema."""

    def with_structured_output(
        self, schema=None, *, method="function_calling",
        include_raw=False, strict=None, **kwargs
    ):
        return super().with_structured_output(
            schema, method=method, include_raw=include_raw,
            strict=strict, **kwargs
        )


def load_project_env(filename=".env", override=True):
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        env_path = directory / filename
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if override or key not in os.environ:
                    os.environ[key] = value
            return env_path
    return None


PROJECT_ENV_PATH = load_project_env(override=True)

# LangSmith tracing is configured before any LangChain objects are created.
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "Q & A chatbot")
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-pro")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
DEEPSEEK_THINKING = os.getenv("DEEPSEEK_THINKING", "disabled")

SUPPORTED_LLM_PROVIDERS = {"openai", "ollama", "gemini", "deepseek"}
if LLM_PROVIDER not in SUPPORTED_LLM_PROVIDERS:
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}. Use one of: {sorted(SUPPORTED_LLM_PROVIDERS)}")

# DeepSeek provides chat models but no embeddings API.
default_embedding_provider = "gemini" if LLM_PROVIDER == "deepseek" else LLM_PROVIDER
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER") or default_embedding_provider
EMBEDDING_PROVIDER = EMBEDDING_PROVIDER.strip().lower()
SUPPORTED_EMBEDDING_PROVIDERS = {"openai", "ollama", "gemini"}
if EMBEDDING_PROVIDER not in SUPPORTED_EMBEDDING_PROVIDERS:
    raise ValueError(f"Unsupported EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER}. Use one of: {sorted(SUPPORTED_EMBEDDING_PROVIDERS)}")


def _get_api_key(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return getpass.getpass(f"Enter your {names[0]}")


class GeminiEmbeddingsOneByOne:
    def __init__(self, embeddings, delay_seconds=0.7, max_retries=5):
        self.embeddings = embeddings
        self.delay_seconds = delay_seconds  # pause between requests to stay under the rate limit
        self.max_retries = max_retries

    def _embed_one(self, text):
        import time
        for attempt in range(self.max_retries + 1):
            try:
                return self.embeddings.embed_documents([text], batch_size=1)[0]
            except Exception as exc:
                # Back off and retry only on quota errors (429 RESOURCE_EXHAUSTED)
                if "RESOURCE_EXHAUSTED" not in str(exc) or attempt == self.max_retries:
                    raise
                wait = 5 * 2 ** attempt
                print(f"Rate limit hit, retrying in {wait}s...")
                time.sleep(wait)

    def embed_documents(self, texts):
        # Gemini embedding models can return one vector per batch with this
        # LangChain version. Chroma expects one vector per document, so embed
        # each chunk separately, with a small delay between requests.
        import time
        vectors = []
        for text in texts:
            vectors.append(self._embed_one(text))
            time.sleep(self.delay_seconds)
        return vectors

    def embed_query(self, text):
        return self.embeddings.embed_query(text)


def ensure_ollama_model_available(model_name):
    import ollama

    try:
        installed_models = {model.model for model in ollama.list().models}
    except Exception as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {OLLAMA_BASE_URL}. "
            "Start Ollama before running this notebook."
        ) from exc

    model_aliases = {name.removesuffix(":latest") for name in installed_models}
    if model_name not in installed_models and model_name not in model_aliases:
        raise RuntimeError(
            f'Ollama model "{model_name}" is not installed. '
            f'Run `ollama pull {model_name}` in a terminal, then restart the notebook kernel.'
        )


def get_embeddings_model():
    if EMBEDDING_PROVIDER == "openai":
        return OpenAIEmbeddings(
            model=OPENAI_EMBEDDING_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if EMBEDDING_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_EMBEDDING_MODEL)
        return OllamaEmbeddings(
            model=OLLAMA_EMBEDDING_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if EMBEDDING_PROVIDER == "gemini":
        embeddings = GoogleGenerativeAIEmbeddings(
            model=GEMINI_EMBEDDING_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
        return GeminiEmbeddingsOneByOne(embeddings)


def get_chat_model():
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(
            model=OPENAI_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_MODEL)
        return ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        return ChatGoogleGenerativeAI(
            model=GEMINI_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
    if LLM_PROVIDER == "deepseek":
        return ChatDeepSeek(
            model=DEEPSEEK_MODEL,
            openai_api_key=_get_api_key("DEEPSEEK_API_KEY"),
            base_url=DEEPSEEK_BASE_URL,
            extra_body={"thinking": {"type": DEEPSEEK_THINKING}},
        )


def message_text(message):
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


print(f"Loaded env from: {PROJECT_ENV_PATH or 'not found'}")
print(f"Using {LLM_PROVIDER} chat model with {EMBEDDING_PROVIDER} embeddings")
if LLM_PROVIDER == "ollama":
    print(f"Ollama chat model: {OLLAMA_MODEL}")
elif LLM_PROVIDER == "gemini":
    print(f"Gemini chat model: {GEMINI_MODEL}")
elif LLM_PROVIDER == "deepseek":
    print(f"DeepSeek chat model: {DEEPSEEK_MODEL}")
print(f"LangSmith tracing: {os.getenv('LANGSMITH_TRACING', 'false')} | project: {os.getenv('LANGSMITH_PROJECT')}")


### 7.2 Vector store content ingestion

Before the system can answer questions across documents, the source material must be loaded into a vector database. This notebook ingests content about Paestum from several sources and formats, so the later retrieval step can search across a small knowledge base instead of a single text.

The loaders read the original files, the splitter breaks them into searchable chunks, the embedding model turns each chunk into a vector, and Chroma stores both the vectors and the original text for later retrieval.


In [ ]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


## Setting up vector database and embeddings

### Splitting and storing documents

The splitter creates chunks of about 500 characters. Smaller chunks make retrieval more precise because a query can match a focused passage instead of a large mixed section. Here the overlap is set to `0` to keep the example simple.

The embedding model is selected through `get_embeddings_model()`, so the same notebook can build the vector store with OpenAI, Ollama, or Gemini embeddings independently of the chat provider. The Chroma collection named `tourist_info` stores the resulting chunks and their embeddings.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=0)
embeddings_model = get_embeddings_model()
vector_db = Chroma(
    collection_name="tourist_info",
    embedding_function=embeddings_model,
)


### Loading the first document sources

Each loader knows how to read one source type: `Docx2txtLoader` reads Word files, `PyPDFLoader` reads PDFs, and `TextLoader` reads plain text. After loading, every document is split and added to the same Chroma collection.

The original version of this notebook also loaded a Wikipedia page. That step is removed here so that ingestion uses only the local files in `data/`.


In [ ]:
word_loader = Docx2txtLoader("data/Paestum/Paestum-Britannica.docx")
word_chunks = text_splitter.split_documents(
    word_loader.load())
vector_db.add_documents(word_chunks)

In [ ]:
pdf_loader = PyPDFLoader("data/Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(
    pdf_loader.load())
vector_db.add_documents(pdf_chunks)

In [ ]:
txt_loader = TextLoader("data/Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(
    txt_loader.load())
vector_db.add_documents(txt_chunks)

### Removing duplication

### Why remove duplication

The ingestion pattern is the same for every loader: load documents, split them into chunks, and add the chunks to Chroma. Pulling that pattern into `split_and_import()` keeps the notebook easier to extend when additional file types or sources are added.


In [ ]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)
     print(f"Ingested chunks created by {loader}")

In [ ]:
word_loader = Docx2txtLoader("data/Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("data/Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("data/Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

## Ingesting Multiple Documents from a Folder (two techniques)

### Ingesting many files from a folder

Loading one file at a time works for a few sources, but a real knowledge base usually contains many documents. The next cells show two approaches: iterate over the folder and choose the loader from the file extension, or use LangChain's `DirectoryLoader` when the optional unstructured dependencies are installed.


### 1) Iterating over all files in a folder

### Loader factory by extension

The loader factory maps file extensions to the proper LangChain loader. This keeps the folder loop generic: it does not need to know how to process DOCX, PDF, or TXT directly; it only asks `get_loader()` to instantiate the correct loader for each file.


In [ ]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [ ]:
import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename) #A Extract the file extension
    file_extension = file_extension.lstrip('.') #B Remove the leading dot from the extension
    
    loader_class = loader_classes.get(
        file_extension) #C Get the loader class from the dictionary
    
    if loader_class:
        return loader_class(filename) #D Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

#### Ingesting the files from the folder (Exercise solution)

### Folder ingestion loop

The loop walks through `CilentoTouristInfo`, skips directories, creates the appropriate loader for each file, and reuses `split_and_import()` to add the content to Chroma. Unsupported extensions are reported instead of stopping the whole ingestion process.


In [ ]:
folder_path = "data/CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file
   
    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

### 2) Ingesting all files with with DirectoryLoader

### DirectoryLoader alternative

`DirectoryLoader` can ingest a whole folder with a glob pattern, but it depends on optional unstructured-document packages whose installation can vary by operating system. The explicit folder loop above is more transparent; this cell shows the shorter alternative for environments where those dependencies are available.


In [ ]:
# ONLY RUN THIS IF YOU HAVE SUCCESFULLY INSTALLED unstructured or langchain-unstructured
# THE INSTALLATION IS OPERATIVE SYSTEM SPECIFIC
# follow LangChain instructions at https://python.langchain.com/v0.2/docs/integrations/providers/unstructured/ or 
# Unstructured instructions at https://docs.unstructured.io/welcome#quickstart-unstructured-open-source-library
folder_path = "data/CilentoTouristInfo"
pattern = "**/*.{docx,pdf,txt}" #A Pattern to match .docx, .pdf, and .txt files

directory_loader = DirectoryLoader(folder_path, pattern) #B Initialize the DirectoryLoader with the folder path and pattern
split_and_import(directory_loader)

### 7.3 Q&A across stored documents

After ingestion, the vector store can be queried across all stored documents. This is the point where the notebook checks whether the chunking and ingestion strategy worked: a question should retrieve relevant passages even when the answer is spread across different files and sources.


## Querying the vector store directly

### Direct vector store query

The direct `similarity_search()` call asks Chroma for the four closest chunks. This does not generate a final answer; it only returns the retrieved documents and metadata. Inspecting these raw results is useful before adding the LLM because it shows what context the RAG chain will receive.


In [ ]:
query = "Where was Poseidonia and who renamed it to Paestum?" 
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

In [ ]:
len(results)

## Asking a question through a RAG chain

### Prompting the RAG answer

The prompt template defines the contract for answer generation. It receives `{context}` from the retriever and `{question}` from the user, then instructs the model to answer only from the retrieved material and to say when it does not know.


In [ ]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end. 
If you don't know the answer, just say that you don't know, 
don't try to make up an answer.
Use three sentences maximum and keep the 
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

### Completing the RAG chain

The RAG chain has three moving parts: the retriever supplies context, `RunnablePassthrough` forwards the original question, and the chat model generates the answer. LCEL connects these pieces with `|`, so the output of the prompt becomes the input to the selected LLM.


In [ ]:
retriever = vector_db.as_retriever()

In [ ]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

In [ ]:
chatbot = get_chat_model()


In [ ]:
# set up RAG chain

rag_chain = {"context": retriever, 
             "question": question_feeder}|rag_prompt|chatbot

### Executing the chain

`execute_chain()` is a thin wrapper around `chain.invoke(question)`. Keeping it as a function makes later examples easier to compare, especially when message history is added in the next section.


In [ ]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [ ]:
question = """Where was Poseidonia and who renamed 
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(message_text(answer))

In [ ]:
print(answer)

### Stateless follow-up question

This follow-up question is intentionally vague. The chain built so far is stateless: it receives only the current question, retrieves context for that question, and calls the LLM. It does not remember that the previous question was about Poseidonia and the Romans, so pronouns such as "they" can be misinterpreted.


In [ ]:
question = """And then, what they do? 
Tell me only if you know. 
Also tell me the source""" 
answer = execute_chain(rag_chain, question)
print(message_text(answer))

### 7.4 Chatbot memory of message history

A chatbot needs message history to handle follow-up questions. Without memory, each question is treated as an isolated request. With memory, the prompt can include previous human and AI messages, giving the model enough conversational context to resolve references such as "they" or "then".


## Chatbot memory of message history

### Memory-enabled RAG chain

The memory-enabled chain switches from a plain `PromptTemplate` to `ChatPromptTemplate.from_messages()`. This lets the prompt include a system instruction, previous chat messages, retrieved context, and the current human question as separate chat roles.

`ChatMessageHistory` stores the conversation. `RunnableLambda(get_messages)` injects the current message history into the prompt each time the chain runs, and `execute_chain_with_memory()` updates the history after each model response.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = get_chat_model()
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    "retrieved_context": retriever, 
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')                                      
    return answer


### Testing the memory-enabled chain

The first question populates the message history with a concrete topic. The second, vaguer follow-up can then use that history to understand what the user is referring to. This demonstrates the difference between a stateless RAG engine and a conversational RAG chatbot.


In [ ]:
question = """Where was Poseidonia and who renamed 
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(message_text(answer))

In [ ]:
question = """And then what did they do? 
Also tell me the source""" 
answer = execute_chain_with_memory(rag_chain, question)
print(message_text(answer))

## Tracing with LangSmith

LangSmith tracing can be configured directly from the project-root `.env` file. The first notebook cell loads those values before any LangChain objects are created, so traces are captured as soon as you run the RAG chain cells.

Add your LangSmith key to `.env`:

```bash
LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```

The notebook sets these defaults automatically when they are not already defined:

```bash
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_PROJECT=Q & A chatbot
```

If `LANGSMITH_API_KEY` is present, the notebook enables tracing with:

```bash
LANGSMITH_TRACING=true
```

After editing `.env`, restart the kernel and run the notebook from the first cell so the environment is loaded before the LangChain chains are built. The RAG chain and memory-enabled RAG chain executions will then appear in the configured LangSmith project.
